# AgentRegistry, end to end: scaffold a dice agent → run it on kagent **and** AWS Bedrock AgentCore

Scaffold an agent with `arctl`, run it locally, publish it to the registry, then deploy the same published agent to two runtimes — Solo Enterprise for kagent (local kind) and AWS Bedrock AgentCore. The two deployments differ only in the Deployment's `runtimeRef`.

> **Kernel:** pick **Bash** (top-right). Engineer setup lives in `setup/` (see `setup/README.md`).

## Connect to the platform

Loads creds, puts `arctl` on the path, mints a catalog token.

In [ ]:
[ -d agentregistry-agentcore-kind ] && cd agentregistry-agentcore-kind; source setup/scripts/connect.sh && rm -rf agentdemo

## 1. Create a new agent project

Scaffolds the agent project into `agentdemo/`. Open it in the Explorer to review the generated code (`roll_die`, `check_prime`).

In [ ]:
arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5

## 2. Walk through the dice agent

In [ ]:
cat agentdemo/agentdemo/agent.py

## 3. Build the agent image

In [ ]:
arctl build ./agentdemo

Run it locally in an interactive chat (use a **terminal** — it's interactive):

```sh
arctl run ./agentdemo
```

## 4. Publish to the catalog

In [ ]:
arctl build ./agentdemo --push

In [ ]:
arctl apply -f agentdemo/agent.yaml

In [ ]:
arctl get agent agentdemo

## 5. Deploy the agent onto kagent (runtime #1)

Binds the agent to the `kind-kagent` runtime. The AgentCore deployment below is identical except for `runtimeRef`.

In [ ]:
envsubst < setup/yaml/deploy-kagent.yaml | arctl apply -f -

In [ ]:
kubectl --context kind-$CLUSTER_NAME -n kagent get pods -l app.kubernetes.io/name=agentdemo-latest-agentdemo

## 6. Talk to the dice agent — through real OIDC

Mints a Keycloak token for `alice` and sends an A2A message. The agent calls `roll_die`, then `check_prime`.

In [ ]:
./setup/scripts/ask.sh "Roll a 20-sided die and tell me whether the result is a prime number."

---
# Deploy the same agent to AWS Bedrock AgentCore (runtime #2)

The same published agent, deployed to AWS. On AgentCore it uses Bedrock Claude via the AWS role (no API key). Requires an AWS account; skip for a local-only run.

## 7. Sign in to AWS

In [ ]:
source setup/scripts/aws-login.sh

## 8. Deploy the same agent to AgentCore

Applies the cross-account IAM role, registers the `BedrockAgentCore` runtime, pushes the image and source, applies the Deployment, and waits for the runtime to reach READY.

In [ ]:
./setup/scripts/agentcore-deploy.sh

## 9. Test the dice agent on AgentCore

In [ ]:
./setup/scripts/ac-invoke.sh "Roll a 20-sided die and tell me whether the result is a prime number."

The same published agent runs on both Solo Enterprise for kagent and AWS Bedrock AgentCore.

## Reset / teardown

```sh
./setup/scripts/reset.sh      # back to start
./setup/scripts/cleanup.sh    # full teardown
```